# LaBraM fine-tuning за MPD-DF

Овој notebook го fine-tune-ира официјалниот pretrained LaBraM-base со O1 канал. Не прави LOSO benchmark. Пред старт избери **Runtime → Change runtime type → GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/eeg-fatigue-labram')
BUNDLE = DRIVE_DIR / 'labram_colab_bundle.tar'
RESULTS_DIR = DRIVE_DIR / 'results' / 'o1_finetune'
WORK_DIR = Path('/content/eeg-fatigue-labram')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file(), f'Прикачи ја архивата во: {BUNDLE}'
print('Bundle:', BUNDLE)
print('Results:', RESULTS_DIR)

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU не е активен. Избери Runtime > Change runtime type > GPU.')

In [ ]:
import subprocess

marker = WORK_DIR / '.bundle_extracted'
if not marker.is_file():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xf', str(BUNDLE), '-C', str(WORK_DIR)], check=True)
    marker.touch()
    print('Bundle extracted.')
else:
    print('Bundle already extracted.')

DATA_ROOT = WORK_DIR / 'data' / 'labram_processed'
TRAIN_SCRIPT = WORK_DIR / 'ml-service' / 'scripts' / 'train_labram_mpd_df.py'
assert len(list(DATA_ROOT.glob('participant_*'))) == 30
assert TRAIN_SCRIPT.is_file()
print('30 participants found.')

In [ ]:
%pip install -q timm==0.9.16 einops==0.8.1

In [ ]:
LABRAM_COMMIT = 'c431221e6cfd23dbfa9950e0180682fb322b0548'
LABRAM_REPO = Path('/content/LaBraM')
if not (LABRAM_REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/935963004/LaBraM.git', str(LABRAM_REPO)], check=True)
subprocess.run(['git', '-C', str(LABRAM_REPO), 'checkout', LABRAM_COMMIT], check=True)
CHECKPOINT = LABRAM_REPO / 'checkpoints' / 'labram-base.pth'
assert CHECKPOINT.is_file() and CHECKPOINT.stat().st_size > 90_000_000
print('Official LaBraM commit:', LABRAM_COMMIT)
print('Checkpoint MB:', round(CHECKPOINT.stat().st_size / 1_000_000, 1))

In [ ]:
import numpy as np
sample_path = DATA_ROOT / 'participant_01' / 'eeg_windows_4ch_5s_200hz_uv.npz'
with np.load(sample_path, allow_pickle=False) as sample:
    print('Example X:', sample['X'].shape, sample['X'].dtype)
    print('Channels:', sample['channels'].tolist())
    print('Sampling rate:', float(sample['sampling_rate']), 'Hz')
    print('Unit:', str(sample['unit']))

## Fine-tuning

Оваа ќелија ги користи испитаниците 01–24 за training, 25–27 за validation и 28–30 само за финален test. `--resume` продолжува од последниот зачуван epoch ако Colab бил прекинат.

In [ ]:
import sys
command = [
    sys.executable, str(TRAIN_SCRIPT),
    '--data-root', str(DATA_ROOT),
    '--labram-repo', str(LABRAM_REPO),
    '--checkpoint', str(CHECKPOINT),
    '--output-dir', str(RESULTS_DIR),
    '--epochs', '15',
    '--patience', '3',
    '--batch-size', '128',
    '--device', 'cuda',
    '--resume',
]
print('Starting LaBraM fine-tuning...')
subprocess.run(command, check=True)

In [ ]:
import json
metrics_path = RESULTS_DIR / 'test_metrics.json'
assert metrics_path.is_file(), 'Training is not complete yet.'
result = json.loads(metrics_path.read_text())
for name in ['test_balanced_accuracy', 'test_sensitivity_recall', 'test_specificity', 'test_f1', 'test_roc_auc']:
    print(f'{name}: {result[name]:.3f}')
print('Best epoch:', result['best_epoch'])
print('Best model:', RESULTS_DIR / 'labram_o1_best.pt')